### Simple Gen AI App

In [3]:
import os

from dotenv import load_dotenv
load_dotenv()

required_variables = [
    "LANGSMITH_API_KEY",
    "LANGSMITH_PROJECT",
]
# LangSmith Tracking
os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2']=os.getenv("LANGCHAIN_TRACING_V2")
os.environ['LANGCHAIN_PROJECT']=os.getenv("LANGCHAIN_PROJECT")


In [4]:
## Data Ingestion from the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [5]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
loader

In [6]:
docs=loader.load()

In [7]:
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and b

In [9]:
### Load data -> Docs -> Divide our text into chunks -> vectors -> Vectors embedding  --> Vector store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [10]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content="Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and bi

In [13]:
from langchain_ollama import OllamaEmbeddings


In [15]:
embeddings=(
    OllamaEmbeddings(model="gemma2:2b")  # by default is use llama2
)
embeddings

OllamaEmbeddings(model='gemma2:2b', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [16]:
from langchain_community.vectorstores import FAISS
vector_store_db=FAISS.from_documents(documents,embeddings)


In [17]:
vector_store_db

In [21]:
## Query from a vector db
query="in a real application you would replace it with a vector search or similar."
result=vector_store_db.similarity_search(query)
result[0].page_content

'function retriever(query: string): string[] {\n    return docs;\n}\n\nconst supportBot = traceable(async function supportBot(question: string): Promise<string> {\n    const context = retriever(question);\n    const systemMessage =\n        "You are a helpful customer support agent. " +\n        "Answer using only the information provided below:\\n\\n" +\n        context.join("\\n");\n    const response = await client.chat.completions.create({\n        model: "gpt-5.4-mini",\n        messages: [\n            { role: "system", content: systemMessage },\n            { role: "user", content: question },\n        ],\n    });\n    return response.choices[0].message?.content ?? "";\n});\n\n(async () => {\n    console.log(await supportBot("How many users can I have on the Starter plan?"));\n})();\n\nCalling support_bot("How many users can I have on the Starter plan?") now produces a trace of the full RAG pipeline.'

In [24]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma2:2b",
    temperature=0
)
print(llm)


metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}} output_version=None model='gemma2:2b' temperature=0.0


In [25]:
#  Retrieval chain, Documents chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    """
)

document_chain=create_stuff_documents_chain(
    llm,prompt
)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, model='gemma2:2b', temperature=0.0)
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [28]:
from langchain_core.documents import Document

document_chain.invoke(
    {
        "input":"you will build a customer support chatbot using retrieval-augmented generation (RAG) and add LangSmith observability",
        "context":[Document(page_content="you will build a customer support chatbot using retrieval-augmented generation (RAG) and add LangSmith observability")]
    }
)


"Based on the provided context, here's how I would approach building a customer support chatbot using RAG and incorporating LangSmith observability:\n\n**1. Understanding the Context:**\n\n* **Customer Support Chatbot:**  The goal is to create a conversational AI that can handle customer inquiries and provide assistance. \n* **Retrieval-Augmented Generation (RAG):** This technique combines retrieval with language models. It allows the chatbot to find relevant information from a knowledge base (KB) based on user queries, then uses this information to generate responses.\n* **LangSmith Observability:**  This is likely a tool for monitoring and analyzing the performance of the RAG-powered chatbot. It provides insights into how well the chatbot is performing, identifies potential issues, and helps optimize its functionality.\n\n**2. Building the Chatbot:**\n\n* **Knowledge Base (KB):** The first step is to create or curate a comprehensive KB containing all relevant information about produc

##### However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [29]:
### Input--->Retriever ---> vectorStoreDB
vector_store_db

In [34]:
retriever=vector_store_db.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [35]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x10fad3230>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
     

In [37]:
### Get the response from the LLM
response=retrieval_chain.invoke({
                           "input": "you will build a customer support chatbot using retrieval-augmented generation (RAG) and add LangSmith observability"
})


In [40]:
response['answer']

'This tutorial explains how to trace an LLM application using LangChain. \n\nHere\'s a breakdown of the key points:\n\n**1. Trace LLM Calls:** The core focus is on tracing the entire process of interacting with the LLM (e.g., "support_bot"). This involves understanding the flow of data and actions within the system.\n\n**2. Tracing Setup:**  The tutorial provides a basic example using LangChain to trace calls, demonstrating how to use `trace` function for this purpose. \n\n**3. A/B Testing:** The tutorial mentions A/B testing as a method for comparing different versions of the LLM application. This involves creating two groups (e.g., "group_A" and "group_B") with different variations, then analyzing their performance to determine which is better. \n\n**4. Drilldown:**  The tutorial explains how to use drill-down functionality in the monitoring interface to investigate specific data points within a trace. This allows users to zoom into details for further analysis.\n\n\n**In summary,** 

In [41]:
response


{'input': 'you will build a customer support chatbot using retrieval-augmented generation (RAG) and add LangSmith observability',
 'context': [Document(id='671343d3-ca25-4e12-a7d5-44c213de1feb', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content="Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewTrac

In [42]:
response['context']

[Document(id='671343d3-ca25-4e12-a7d5-44c213de1feb', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content="Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & en